# ARC-05: Simulation Configuration

## Purpose
Define the configuration contract for deterministic Earth-orbit spacecraft simulation.

## Scientific conventions
- Coordinate frame: Earth-Centered Inertial (ECI)
- Position: 3D NumPy array in metres (m)
- Velocity: 3D NumPy array in metres per second (m/s)
- Time: seconds (s)
- Gravitational parameter: m^3/s^2
- Initial baseline: Newtonian Earth two-body gravity
- Proposed numerical integrator: fixed-step RK4

## Configuration fields
1. Initial position
2. Initial velocity
3. Earth gravitational parameter
4. Simulation duration
5. Integration time step
6. Integrator selection
7. Collision threshold

## Scientific safeguards
- Reject non-finite values.
- Require three-dimensional position and velocity.
- Enforce SI unit consistency.
- Do not invent physical constants or safety thresholds.

## Pending decisions
- Authoritative source and value for Earth's gravitational parameter.
- Authoritative collision threshold and its interpretation.
- Configuration serialization contract.

## Earth Reference Constants

The initial Earth model uses the WGS 84 defining parameters published
by the US National Geospatial-Intelligence Agency (NGA).

- Earth gravitational parameter: 3.986004418e14 m^3/s^2
- WGS 84 semi-major axis: 6378137.0 m
- WGS 84 inverse flattening: 298.257223563

The semi-major axis is used as a spherical reference collision boundary
in this educational baseline. It is not a terrain, geoid, or atmospheric
boundary.

### References

1. NGA, World Geodetic System 1984:
   https://earth-info.nga.mil/index.php?action=wgs84&dir=wgs84

2. NASA, Earth Fact Sheet:
   https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html

### Frame distinction

ECI is the inertial coordinate frame for spacecraft state vectors.
WGS 84 supplies the Earth reference figure and constants. These are
different concepts and must not be treated as interchangeable.


## Timestep Policy

The simulation supports a final partial timestep. The propagation must end exactly at the configured simulation duration, even when the duration is not an integer multiple of the nominal timestep. The final integration step is shortened when necessary.

For duration T and nominal timestep dt:

- Number of steps: ceil(T / dt)
- All steps except possibly the last use dt.
- The last step uses the remaining simulation time.


In [1]:

import math
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

EARTH_MU_M3_S2 = 3.986004418e14
EARTH_REFERENCE_RADIUS_M = 6_378_137.0
WGS84_INVERSE_FLATTENING = 298.257223563


class SimulationConfig(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        frozen=True,
        strict=True,
        allow_inf_nan=False
    )

    initial_position_m: tuple[float, float, float]
    initial_velocity_mps: tuple[float, float, float]

    gravitational_parameter_m3_s2: float = Field(
        default=EARTH_MU_M3_S2,
        gt=0
    )

    simulation_duration_s: float = Field(gt=0)
    time_step_s: float = Field(gt=0)

    integrator: Literal["RK4"] = "RK4"
    reference_frame: Literal["ECI"] = "ECI"

    collision_radius_m: float = Field(gt=0)

    @field_validator("initial_position_m", "initial_velocity_mps", mode="before")
    @classmethod
    def validate_vector_inputs(cls, value):
        if isinstance(value, (str, bytes, bytearray)):
            raise ValueError("Position and velocity components must be numeric")
        return value

    @model_validator(mode="after")
    def validate_physical_values(self):
        values = (
            *self.initial_position_m,
            *self.initial_velocity_mps,
            self.gravitational_parameter_m3_s2,
            self.simulation_duration_s,
            self.time_step_s,
            self.collision_radius_m,
        )

        if not all(math.isfinite(value) for value in values):
            raise ValueError("All configuration values must be finite")

        return self


# Example configuration for validation
example_config = SimulationConfig(
    initial_position_m=(7_000_000.0, 0.0, 0.0),
    initial_velocity_mps=(0.0, 7_500.0, 0.0),
    simulation_duration_s=600.0,
    time_step_s=10.0,
    collision_radius_m=EARTH_REFERENCE_RADIUS_M,
)

print("Configuration created successfully")
print(example_config.model_dump())


Configuration created successfully
{'initial_position_m': (7000000.0, 0.0, 0.0), 'initial_velocity_mps': (0.0, 7500.0, 0.0), 'gravitational_parameter_m3_s2': 398600441800000.0, 'simulation_duration_s': 600.0, 'time_step_s': 10.0, 'integrator': 'RK4', 'reference_frame': 'ECI', 'collision_radius_m': 6378137.0}


In [2]:

from pydantic import ValidationError

def expect_invalid(**overrides):
    data = example_config.model_dump()
    data.update(overrides)

    try:
        SimulationConfig(**data)
    except ValidationError:
        return True

    raise AssertionError("Invalid configuration was accepted")


assert expect_invalid(initial_position_m=(1.0, 2.0))
assert expect_invalid(simulation_duration_s=float("nan"))
assert expect_invalid(time_step_s=float("inf"))
assert expect_invalid(collision_radius_m=0.0)
assert expect_invalid(unknown_option=True)
assert expect_invalid(initial_position_m=("7000000", 0.0, 0.0))
assert expect_invalid(initial_velocity_mps=(False, 0.0, 0.0))
assert expect_invalid(initial_position_m=(7_000_000.0, 1j, 0.0))

try:
    example_config.time_step_s = 20.0
    raise AssertionError("Frozen configuration was modified")
except ValidationError:
    pass

print("Invalid dimensions: REJECTED")
print("NaN and Infinity: REJECTED")
print("Non-positive collision radius: REJECTED")
print("Unknown fields: REJECTED")
print("Mutation of frozen model: REJECTED")
print("All negative validation checks: PASSED")


Invalid dimensions: REJECTED
NaN and Infinity: REJECTED
Non-positive collision radius: REJECTED
Unknown fields: REJECTED
Mutation of frozen model: REJECTED
All negative validation checks: PASSED
